# Task 2 – exact attention, MQAR, KV-cache memory
2A ≈ 20–40 min, 2B ≈ 1–1.5 h (42 runs; first run per (K, regime, seed) also generates the training stream), 2C ≈ 1–1.5 h.

### Setup (run every session)
Results and large artefacts go to Google Drive so a disconnected runtime loses nothing; every step is resumable.
Use an **A100** (or L4) runtime with High-RAM: *Runtime → Change runtime type*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_URL = 'https://github.com/adilawan1/ATML-PA4.git'   # <- edit once
DRIVE = '/content/drive/MyDrive/atml-pa4'
os.makedirs(DRIVE, exist_ok=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less fragmentation (helps on 16 GB T4)
os.environ['PA4_RESULTS'] = f'{DRIVE}/results'
os.environ['PA4_CKPT'] = f'{DRIVE}/checkpoints'
os.environ['PA4_DATA'] = f'{DRIVE}/atml-pa4-datasets'   # on Drive so data/local_source_revisions.json persists
if not os.path.exists('/content/atml-pa4'):
    !git clone -q $REPO_URL /content/atml-pa4
%cd /content/atml-pa4
!git pull -q
!pip install -q -r requirements.txt
!bash scripts/get_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/record_env.py | tail -1

### If the runtime disconnects or times out
Nothing is lost: every result, partial result and training checkpoint is written to Google Drive as it is produced
(atomic writes, fsynced logs). Reconnect, re-run the **Setup** cell, run the progress cell below, then re-run the
cell that was interrupted: it resumes where it stopped (MQAR runs and Task-1 conditions restart from their beginning,
everything longer resumes mid-step). Keep this tab open; Colab Pro can still disconnect idle or very long sessions.

In [ ]:
!python scripts/progress.py   # what is finished / where each step will resume

In [ ]:
!python scripts/task2a.py            # dense / tiled / SDPA benchmark, tile sweep, edge cases

In [ ]:
!python scripts/task2b.py verify && python scripts/task2b.py costs

In [ ]:
!python scripts/task2b.py train      # B1 + B2: 13 configurations x 3 seeds (resumable)

### B3 intervention (run only after looking at B1/B2)
1. `python analysis/task2_report.py` and look at the linear/dilated/window results and curves.
2. If the observations still motivate the gated local+linear hybrid, keep (or edit) `B3_intervention.prediction_before_evaluation` in `configs/task2.yaml` and **commit + push it before running the cell below**, so the git history timestamps the prediction before any B3 result exists.
3. Run the B3 cell.

In [ ]:
!python analysis/task2_report.py >/dev/null; python scripts/task2b.py b3   # K=16 mixed x 3 seeds

In [ ]:
!python scripts/task2b.py sweep      # forward-only T in {256,1024,4096,8192}

In [ ]:
!python scripts/task2c.py eval       # C1 checks + 5 policies + FP16 on 180 prompts (resumable)

In [ ]:
!python scripts/task2c.py bench      # C5 packed encode/decode + decode-step latency

In [ ]:
!python analysis/task2_report.py